In [0]:
%sql
DESCRIBE TABLE project_staging.stg_ais_gaps;

In [0]:
%sql
SELECT COUNT(*) AS null_vessel_ids
FROM project_staging.stg_vessels
WHERE vessel_id IS NULL;

In [0]:
import importlib
import schemas.silver_schemas as ss

importlib.reload(ss)

tables = [
    "stg_port_visits",
    "stg_vessel_events",
    "stg_rejects",
    "stg_ais_gaps",
    "stg_vessels",
]

for table in tables:
    print(f"\n{'=' * 60}")
    print(f"CHECKING: {table}")
    print("=" * 60)

    expected = ss.SILVER_SCHEMAS[table]
    expected_fields = {
        field.name: field
        for field in expected.fields
    }

    try:
        actual = spark.table(f"project_staging.{table}").schema
        actual_fields = {
            field.name: field
            for field in actual.fields
        }

        missing = sorted(set(expected_fields) - set(actual_fields))
        extra = sorted(set(actual_fields) - set(expected_fields))

        type_mismatches = []
        nullability_mismatches = []

        for name in sorted(set(expected_fields) & set(actual_fields)):
            exp = expected_fields[name]
            act = actual_fields[name]

            if exp.dataType != act.dataType:
                type_mismatches.append(
                    (name, exp.dataType.simpleString(),
                     act.dataType.simpleString())
                )

            if exp.nullable != act.nullable:
                nullability_mismatches.append(
                    (name, exp.nullable, act.nullable)
                )

        print("Missing columns:", missing)
        print("Extra columns:", extra)
        print("Type mismatches:", type_mismatches)
        print("Nullability mismatches:", nullability_mismatches)

        passed = not (
            missing or extra or type_mismatches or nullability_mismatches
        )
        print("RESULT:", "PASS" if passed else "MISMATCH")

    except Exception as e:
        print("RESULT: TABLE CHECK FAILED")
        print("Reason:", str(e))

In [0]:
vessel_id STRING NOT NULL,

In [0]:
%python
%pip install pytest

In [0]:
%python
import sys
import pytest

sys.dont_write_bytecode = True  # /Workspace doesn't support __pycache__ dirs

# Drop stale cached copies so pytest picks up the latest file edits
for mod in [m for m in sys.modules if m in ("test_vessel_events_logic", "silver.vessel_events_logic")]:
    del sys.modules[mod]

exit_code = pytest.main([
    "-v",
    "-p", "no:cacheprovider",
    "--assert=plain",
    "tests/test_vessel_events_logic.py"
])

print("Pytest exit code:", exit_code)